# Managed statistical analysis

Use this runner for CPU or GPU jobs. It executes a callable in a disposable process, checks live RAM every 0.2 seconds, applies supported GPU allocator limits, and stops on timeout. Only one managed analysis or Dask pool can run per shared home at once. Docker caps cover all notebook processes together.

Edit **analysis_settings.py** to choose the backend and budget. GPU packages are optional; a missing package or unavailable selected GPU fails explicitly. Import GPU packages inside the callable so setup occurs in the worker.

In [ ]:
from labkit import run_analysis
import runpy
ANALYSIS = runpy.run_path('analysis_settings.py')['ANALYSIS']
ANALYSIS

In [ ]:
def summarize():
    import numpy as np
    data = np.random.default_rng(42).normal(size=100_000)
    return {'mean': float(data.mean()), 'std': float(data.std())}

# Set True to execute the example.
RUN_EXAMPLE = False
if RUN_EXAMPLE:
    result = run_analysis(summarize, **ANALYSIS)
    print(result)

## GPU example

After installing a driver-compatible CuPy package, choose `backend="cupy"` in analysis_settings.py. The runner sets the device and allocator limit before calling your function. The Compose GPU override exposes only LAB_GPU_DEVICE (default host GPU 0). Return CPU-side scalars or small arrays. VRAM limits apply to the selected framework allocator; GPU compute utilization is not percentage-capped. Use a dedicated GPU or a host-provisioned MIG slice for isolation.

In [ ]:
def summarize_gpu():
    import cupy as cp
    values = cp.arange(1_000_000, dtype=cp.float32)
    return {'mean': float(values.mean().get())}

# Only execute after selecting and installing the cupy backend:
# result = run_analysis(summarize_gpu, **ANALYSIS)

## Limits and failure behavior

If a job exceeds its sampled RAM ceiling or timeout, its process group is killed and an exception is returned to the notebook. Framework VRAM exhaustion is returned as a job error. The kernel stays alive under normal managed failures. A rapid allocation can outrun monitoring; Docker is the hard RAM boundary and can kill a notebook process/container on exhaustion.

Pass paths for large datasets, keep return values small, and use chunked processing. The input/output transfer cap defaults to 64 MiB; worker file writes also have that per-file limit. Arbitrary notebook cells or custom CUDA allocators can bypass Python guards. This is not a multi-tenant security boundary or a promise that hardware/driver failures cannot affect the host.